# Performance profiling for anemoi (1 hour)

**Make your model faster. Make your model fit in GPU memory.**

In this session, you measure the GPU memory and the speed of the deterministic AIFS-style model from the training session.
You use the same toy dataset (O48), and you start the runs from this notebook, as in the training session.

| Part | Subject | Time |
|---|---|---|
| 1 | Overview: optimisations (Triton GraphTransformer, gradient checkpointing, mapper chunking) | 5 min |
| 2 | Overview: parallelisation (data, model sharding, ensemble) | 5 min |
| 3 | What is in the GPU memory? | 5 min |
| 4 | Allocated and reserved memory | 10 min |
| 5 | The memory snapshot: 4 and 16 mapper chunks | 15 min |
| 6 | Runtime profiling: Triton, PyG and data loader workers | 20 min |

You need a web browser on your laptop to open the memory snapshot and the profiler trace.

This notebook is based on the [anemoi performance optimisation guide](https://anemoi.readthedocs.io/projects/training/en/latest/user-guide/performance-optimisation.html).
For more exercises, refer to the long notebook `perf_profiling.ipynb` in this folder.

## 0. Setup

Run the cell below. It shows the name of your GPU and its total memory.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import shutil
from pathlib import Path

import pandas as pd
import torch
from omegaconf import OmegaConf

import profiling_helpers as ph

# The toy dataset of the training session. On the course server nothing needs to change here.
DATASET = "/datasets/aifs-ea-an-oper-0001-mars-o48-2010-2022-6h-v1-toy.zarr"

# Settings for all the runs in this notebook (Hydra "overrides", as in the training session)
OVERRIDES = [f"system.input.dataset={DATASET}"]

assert torch.cuda.is_available(), "No GPU found. Make sure that you start this notebook on a GPU machine."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:        {gpu.name}")
print(f"GPU memory: {gpu.total_memory / 1e9:.1f} GB")
print(f"Outputs:    {ph.OUTPUT_ROOT}")

All the runs use the config `configs/profiling_o48.yaml`. It is the model of the training session (`deterministic_o48`), with these changes:

* 512 channels (not 256) and a batch of 16 samples (not 2). Then the GPU sets the speed, and the effects are easier to see.
* each run does only **50 training steps**, and does not save checkpoints, plots or MLflow logs
* the **benchmark profiler** (`diagnostics.benchmark_profiler`) is on

Look at the end of the config:

In [ ]:
config = ph.load_config(overrides=OVERRIDES)
print(OmegaConf.to_yaml(config.diagnostics.benchmark_profiler))

| Setting | What it does |
|---|---|
| `memory.enabled` | Records a **profiler trace** (a `*.pt.trace.json` file) for `steps` steps, after `warmup` steps. |
| `snapshot.enabled` | Records a **memory snapshot** (`memory_snapshot.pickle`) for `steps` steps, after `warmup` steps. |
| `time.enabled` | Writes a report of the time for each part of the training loop. |
| `speed.enabled` | Writes a report of the throughput (training steps for each second, it/s). |

To use the profiler, you start anemoi with `anemoi-training profile` (not `anemoi-training train`). In a terminal:

```
anemoi-training profile --config-path=$PWD/configs --config-name=profiling_o48
```

In this notebook, `ph.run_experiment(name, overrides)` does this for you, and writes the output into `~/anemoi-output/profiling/<name>`.

---
## 1. Overview: optimisations (5 min)

The AIFS has an **encoder**, a **processor** and a **decoder**. The encoder and the decoder are also called the **mappers**.

```
  data grid (O48: 10,944 points)
        │   encoder: many edges, data grid → hidden mesh          (large activations)
        ▼
  hidden mesh (2,562 nodes)
        │   processor: 8 GraphTransformer layers on the hidden mesh
  hidden mesh
        │   decoder: many edges, hidden mesh → data grid          (large activations)
        ▼
  data grid (O48: 10,944 points)
```

Three settings change the speed and the memory of this model.

### 1.1 The Triton GraphTransformer (`graph_attention_backend`)

A GraphTransformer layer calculates attention along the **edges** of the graph.
For each edge, it calculates an attention weight, and then it adds the messages of all the edges that go to a node.

Anemoi supports two implementations of the same algorithm.

* With the **PyG** backend (PyTorch Geometric), each part of this calculation is one GPU kernel
  (collect the values for each edge, multiply, softmax, multiply, add for each node).
  Each kernel writes a large tensor (one value for each edge and each attention head) into the GPU memory,
  and the next kernel reads it again. This backend is implemented in PyTorch so it is portable to CPU, macbooks etc. Additionally it uses 'atomics' so it is not deterministic on GPU.
* With the **Triton** backend, anemoi uses its own GPU kernel, written in [Triton](https://triton-lang.org).
  Triton is like a mix of    Python and CUDA, allowing us to write performant code without requiring it to be manually compiled.
  This kernel does all the parts in one step ("kernel fusion"). It does not write the tensors for each edge into the GPU memory.
  Thus, it is **faster** and it uses **less memory**. Additionally, it does not use atomics, therefore it is **deterministic**.

```yaml
model:
  encoder:   
    graph_attention_backend: triton   # or pyg
  processor: 
    graph_attention_backend: triton
  decoder:  
    graph_attention_backend: triton
```

### 1.2 Gradient checkpointing

During the forward pass, PyTorch keeps the intermediate results (the **activations**), because the backward pass needs them.
Usually, the activations use most of the GPU memory. We will see an example of this later in the memory profiling section.

With **gradient checkpointing**, PyTorch keeps only the input of a block (this input is the 'checkpoint'). During the backward pass, it calculates the activations of the block again.
This uses **less memory**, but **more calculation** (the forward pass of the block runs two times, once during the forward pass and once again during the backward pass).

By default anemoi uses gradient checkpointing in the encoder, the processor and the decoder. This can be controlled via the config.

**Question:** How much faster would your model run without gradient checkpointing?

<details>
<summary>Answer</summary>
approximately 33% faster.
With gradient checkpointing you do 2 forwards and a backward.
Without gradient checkpointing you 1 forward and a backward.
</details>

### 1.3 Mapper chunking (`num_chunks`)

The mappers have many edges (one or more for each grid point), thus their activations are large.
With `num_chunks: N`, anemoi divides the edges of a mapper into N parts (**chunks**), and calculates one chunk after the other.
Each chunk is checkpointed. Thus, at one time, the GPU keeps the activations of **one chunk only**.

```
num_chunks: 1   [████████████████████████████████]   one large peak
num_chunks: 4   [████████] [████████] [████████] [████████]   4 smaller peaks
```

More chunks: less memory, but more (smaller) GPU kernels, thus usually a little slower. You see this in part 5.

```yaml
model:
  encoder: 
    num_chunks: 4
  decoder: 
    num_chunks: 4
```

---
## 2. Overview: parallelisation (5 min)

In this session, you have one GPU. Anemoi is typically run over many more. anemoi has three types of parallelisation, which can be mixed and matched. 

**Question:** Do you have any ideas why you might want to train across multiple GPUs rather then just 1?
<details>
<summary>Answer</summary>

Some reasons:
Less time required to train the same models.
Train larger models which dont fit on a single GPU
</details>

**Question:** Can you think of any ways we could split work across multiple GPUs? 
<details>
<summary>Answer</summary>
Some ideas:
We could split the input samples across multiple GPUs.
When training ensemble models, we could run 1 ensemble member per GPU.
We could divide the source grid and attention heads across multiple GPUs.
</details>

### 2.1 Data parallelism

Each GPU has a **full copy** of the model, and calculates a **different sample** of the batch.
After the backward pass, the GPUs calculate the average of their gradients (an "all-reduce"). Then all the copies stay the same. This all-reduce is overlapped with the computation of the backward pass, therefore data parallelism is **practically free** in most cases. 

* It makes the model converge **faster** (more samples computed in parallel for each).
* The it/s is unchanged (this is a **per model** value)
* It does **not** decrease the memory for each GPU.
* The effective batch size increases: `batch_size × number of model copies`.

```yaml
system:
  hardware:
    num_nodes: 2
    num_gpus_per_node: 4     # 8 GPUs, thus 8 copies of the model
```

### 2.2 Model sharding (edge sharding)

When one sample does not fit in the memory of one GPU, anemoi divides **one copy of the model** between more GPUs.
Each GPU keeps a full copy of the weights, but it calculates only a part of the activations:

* in the **mappers**, each GPU calculates a part of the **edges** (`shard_strategy: edges`)
* in the **processor**, each GPU calculates a part of the layers or of the attention heads

The GPUs must communicate during the forward and the backward pass. 

* It makes the model training faster, as the amount of computation per GPU decreases
* Eventually communication will grow to such an extent that runtime will begin to increase again. We say the model has **stopped scaling**
* It **decreases** the activation memory for each GPU.
* It does **not** decrease the memory of the model state (weights, gradients, optimiser).

```yaml
system:
  hardware:
    num_gpus_per_model: 4   # one model copy on 4 GPUs
```

### 2.3 Ensemble parallelism

An ensemble model (e.g. AIFS-ENS, trained with the CRPS loss) calculates many **ensemble members** for each sample. You saw in the ensemble notebook that running multiple ensemble members on the same GPU can get very slow. With ensemble parallelism, anemoi divides the ensemble members of one sample across GPUs. 
The CRPS loss needs all the members, thus the GPUs collect the members of the other GPUs before the loss (an "all-gather" communication step).

```yaml
system:
  hardware:
    num_gpus_per_ensemble: 4   # the members of one sample on 4 GPUs
training:
  ensemble_size_per_device: 2  # 2 members on each GPU, thus 8 members
```

### Summary

| Type | What each GPU has | Decreases memory? | Setting |
|---|---|---|---|
| Data | full model, different samples | no | `num_gpus_per_node`, `num_nodes` |
| Model (edge) sharding | full weights, part of the edges / nodes of one sample | yes (activations) | `num_gpus_per_model` |
| Ensemble | full model, part of the members of one sample | yes (fewer members on each GPU) | `num_gpus_per_ensemble` |

**Question:** You have 16 GPUs, `num_gpus_per_model: 4`. How many copies of the model are there? What is the effective batch size?

<details>
<summary>Answer</summary>

16 / 4 = 4 copies of the model (data parallel). The effective batch size is 4 (4 copies of the model * local batch size 1).
</details>

**Question:** You are running an ensemble model with `num_gpus_per_ensemble: 4` and an effective batch size of 16. How many GPUs will you need?

<details>
<summary>Answer</summary>
    
4 gpus per ensemble samble * 16 copies of the model via data-parallelism => 64 GPUs.
</details>

---
## 3. What is in the GPU memory? (5 min)

During training, the GPU memory used by PyTorch contains these items:

| Item | Size | Changes during a step? |
|---|---|---|
| **Model weights** | 4 bytes for each parameter (float32) | no |
| **Gradients** | 4 bytes for each parameter | yes: made in the backward pass |
| **Optimiser state** (AdamW) | 8 bytes for each parameter (two values) | no |
| **Input batch** | batch size × input times × variables × grid points × 4 bytes | yes |
| **Activations** | depends on the model, the grid, `num_chunks` and checkpointing | yes: they increase in the forward pass and decrease in the backward pass |

The first three items are the **model state**: approximately 16 bytes for each parameter. They stay in the memory for all of the training. 

**Note** some memory elements are proportional to the number of parameters, others are proportional to the number of grid points.

Calculate the size of the input batch of our model:

In [ ]:
from anemoi.datasets import open_dataset

ds = open_dataset(DATASET)
n_variables, n_points = ds.shape[1], ds.shape[3]  # 23 variables, 10 944 grid points (O48)
batch = config.dataloader.batch_size.training
inputs = config.training.multistep_input + 1  # the input times and the target

batch_bytes = batch * inputs * n_variables * n_points * 4
print(f"{batch} samples x {inputs} times x {n_variables} variables x {n_points} points x 4 bytes = {batch_bytes / 1e6:.1f} MB")

The input batch is very small. Thus, almost all of the memory is the model state and the activations.


---
## 4. Allocated and reserved memory (10 min)

PyTorch does not give memory back to the GPU when it deletes a tensor. It keeps the memory in a **cache**, and it uses it again for the next tensor.
This is faster, because to get new memory from the GPU (`cudaMalloc`) is slow.

* **Allocated** memory: the memory that the tensors use now.
* **Reserved** memory: the allocated memory **plus** the cache. This is the memory that PyTorch has taken from the GPU.
  `nvidia-smi` shows the reserved memory (plus the CUDA context).

A small example:

In [ ]:
def show(label):
    print(f"{label:28s} allocated {torch.cuda.memory_allocated() / 1e9:5.2f} GB   reserved {torch.cuda.memory_reserved() / 1e9:5.2f} GB")

show("start")
x = torch.empty(int(1e9), dtype=torch.uint8, device="cuda")  # 1 GB
show("after a 1 GB tensor")
del x
show("after `del x`")
torch.cuda.empty_cache()
show("after `empty_cache()`")

After `del x`, the allocated memory decreases, but the reserved memory stays the same. Only `empty_cache()` gives the memory back to the GPU.


**NOTE** because of the Pytorch Caching memory allocator, using something like *nvidia-smi* or *nvtop* to measure peak memory usage can be unreliable when you want to measure how much memory your model uses.

**Question:** Why can *nvidia-smi* give inaccurate results?

<details>
<summary>Answer</summary>
Because *nvidia-smi* reports the peak **reserved** memory used by PyTorch, not the **allocated** memory.
</details>


The difference between reserved and allocated memory can also be **fragmentation**: free memory in small pieces between tensors.
A large tensor needs one continuous piece. Thus, you can get `CUDA out of memory`, although the reserved memory has sufficient free space.
The setting `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` decreases fragmentation.

Below you can see an out of memory error due to memory fragmentation. The GPU has **4.11 GB** of memory and tried to allocate **3.15 GB**. This should fit, but fails due to memory fragmentation. Setting this 'expandable_segments' env variable can prevent this. Usually you can just *set and forget* this env variable, it does not have a performance cost and works on most systems.
```
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 3.15 GiB. GPU 0 has a total capacity of 39.56 GiB
of which 6.80 GiB is free. Including non-PyTorch memory, this process has 36.61 GiB memory in use.
Of the allocated memory 31.66 GiB is allocated by PyTorch, and 4.11 GiB is reserved by PyTorch but unallocated.
If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
to avoid fragmentation.
```

In anemoi you can use the `diagnostics.print_memory_summary` setting to get an accurate measure of the peak **allocated** memory.

Now we train the anemoi model for 20 steps (with `anemoi-training train`, without the profiler).
`diagnostics.print_memory_summary=True` prints `torch.cuda.memory_summary()` at the end of the training.
This takes approximately 1 to 2 minutes. The first run also makes the graph.

In [ ]:
ph.run_experiment(
    "memory_summary",
    OVERRIDES + ["diagnostics.print_memory_summary=True", "training.max_steps=20"],
    command="train",
)

Find the memory summary at the end of the output. Look at the rows **Allocated memory** and **GPU reserved memory**, and at the columns:

* **Cur Usage**: at the end of the training
* **Peak Usage**: the maximum during the training

**Questions:**

1. What is the peak allocated memory? What is the peak reserved memory? Why are they different?
2. Which value must be less than the GPU memory, so that the run does not fail?

<details>
<summary>Answers</summary>

1. The peak reserved memory is larger than the peak allocated memory. The difference is the cache of PyTorch and fragmentation.
2. The allocated memory. If PyTorch cannot allocate more memory, the run fails with `CUDA out of memory`.
</details>

---
## 5. The memory snapshot (15 min)

The memory summary gives only a few numbers. A **memory snapshot** paints a picture.

the **memory snapshot** records each allocation and each release of GPU memory, with the line of Python code that made it. We can then visualise this data and interactively explore where our model uses memory.
anemoi enables this setting with `diagnostics.benchmark_profiler.snapshot`. 

We do two runs, with 4 mapper chunks (the default) and with 16 mapper chunks.
We switch off the profiler trace (`memory.enabled=False`), because we do not need it here.
Each run takes approximately 1 to 2 minutes.

**Make a prediction:** with 16 chunks, does the peak memory increase or decrease? Does the shape of the memory change?

In [ ]:
def mapper_chunks(n):
    return [f"model.encoder.num_chunks={n}", f"model.decoder.num_chunks={n}"]

SNAPSHOT_ONLY = ["diagnostics.benchmark_profiler.memory.enabled=False"]

snapshots = {}
snapshots["chunks_4"] = ph.run_experiment("chunks_4", OVERRIDES + SNAPSHOT_ONLY + mapper_chunks(4))

In [ ]:
snapshots["chunks_16"] = ph.run_experiment("chunks_16", OVERRIDES + SNAPSHOT_ONLY + mapper_chunks(16))

The cell below copies the two snapshots into the folder `snapshots/`, next to this notebook.

In [ ]:
Path("snapshots").mkdir(exist_ok=True)
for name, profiler_dir in snapshots.items():
    target = Path("snapshots") / f"{name}.pickle"
    shutil.copy(profiler_dir / "memory_snapshot.pickle", target)
    print(f"{target}  ({target.stat().st_size / 1e6:.0f} MB)")

### 5.1 Open the snapshots in the memory viewer

1. In the JupyterLab file browser (on the left), open the folder `snapshots`. Right-click `chunks_4.pickle` and select **Download**. Do the same for `chunks_16.pickle`. The files are approximately 40 MB and 100 MB.
2. On your laptop, open [https://pytorch.org/memory_viz](https://pytorch.org/memory_viz).
3. Drag `chunks_4.pickle` into the web page. Make sure that **Active Memory Timeline** is selected (at the top).
4. Open a second browser tab with the same page, and drag `chunks_16.pickle` into it.

> The file stays on your laptop. The viewer does not send the file to a server.

In the **Active Memory Timeline**, each coloured band is one tensor. The horizontal axis is the time, the vertical axis is the allocated memory. If you see a lot of blue bands, you can increase the level of detail to max to resolve all tensors.

* Move the mouse over a band. The viewer shows the size of the tensor and the Python code that allocated it (the "stack trace").
* Click on a band to keep its stack trace at the bottom of the page.
* Drag on the timeline to zoom in. Click the left mouse button to zoom out.

**Note** the snapshot shows multiple steps. Since memory is typically unchanged across steps, it is usually very helpful to zoom in to a single step

**Exercise:**

1. The snapshot has 16 training steps. Drag on the timeline to zoom in to one step. Find the forward pass (the memory increases) and the backward pass (the memory decreases).
2. Find the memory at the bottom that stays the same all the time. What is it?
3. In `chunks_4`, find the 4 chunks of the encoder at the start of the forward pass. Use the stack trace: look for `encoder`, `mapper` or `checkpoint`.
4. Now look at `chunks_16`. How many chunks do you see? Are they smaller?

<details>
<summary>Click for an example snapshot (O96, 4 mapper chunks and 4 processor chunks)</summary>

![memory snapshot](images/mem_snapshot_anemoi_o96_4mc4pc.png)
</details>

### 5.2 Compare the two runs

The helper functions read the snapshots and plot the allocated memory over time:

In [ ]:
ph.plot_memory_timelines(snapshots, title="4 and 16 mapper chunks")
ph.summary_table(snapshots)

**Questions:**

1. How much did the peak memory decrease from 4 to 16 chunks?
2. Is the peak in the mappers or in the processor? (Look at the stack trace of the largest bands at the peak.)
3. Did the throughput change?

<details>
<summary>Answers</summary>

1. Almost nothing (we measured 4.3 GB for both). The mapper peaks are smaller with 16 chunks, but they are not the highest part of the step.
2. At O48, the grid is small, thus the mapper activations are small. The peak is in the **processor**.
   Thus, more mapper chunks do not decrease the peak.
   **A setting that saves memory is only useful if it changes the part of the model that makes the peak.**
3. The throughput is much lower with 16 chunks (on this machine: approximately 0.79 it/s with 4 chunks, 0.69 it/s with 16 chunks),
   because there are more and smaller GPU kernels. Thus, here 16 chunks have a cost and no benefit.

**Note** increasing the mappper chunks too far can decrease performance, as we see here. Typically `1,2,4,8` mapper chunks are performance neutral. `16,32` mapper chunks can have a performance cost. For chunk counts greater then `32` we typically see no further memory savings, due to the cost of chunking being larger then whats inside the chunk.

If you have time: run `mapper_chunks(1)`. With 1 chunk, the mappers make the peak, and the peak increases (from 6.2 GB to 8.8 GB).

</details>

Additionally, you could increase the number of processor chunks to 16 `model.processor.num_chunks=16`, then you should see peak memory change when you change the mapper chunks

---
## 6. Runtime profiling (20 min)

In this section, we walk through how to measure the speed of your anemoi model.

First we show some examples of different configurations, each configuration will have a different performance. This should hopefully give you a flavour of the different factors affecting your models performance.

`pyg` uses a slower implementation of the *Graph Transformer* algorithm. `workers_1` uses less dataloader workers.


| Run | Attention backend | Data loader workers |
|---|---|---|
| `baseline` | Triton | 8 |
| `pyg` | PyG | 8 |
| `workers_1` | Triton | 0 |

The **data loader workers** are CPU processes. They read the samples from the disk and prepare the next batches, while the GPU trains. If you have too few, then the GPU must wait for the CPU to fetch more data. This slows down your training?

The runs record a **profiler trace** of 2 training steps (steps 5 and 6). We switch off the memory snapshot (`snapshot.enabled=False`), because it makes the training slower.
Each run takes approximately 1 to 2 minutes.

**Make a prediction:** which run is the fastest? Which run is the slowest?

In [ ]:
TRACE_ONLY = ["diagnostics.benchmark_profiler.snapshot.enabled=False"]
PYG = [f"model.{part}.graph_attention_backend=pyg" for part in ["encoder", "processor", "decoder"]]

runs = {}
runs["baseline"] = ph.run_experiment("baseline", OVERRIDES + TRACE_ONLY + ["dataloader.num_workers.training=8"])

In [ ]:
runs["pyg"] = ph.run_experiment("pyg", OVERRIDES + TRACE_ONLY + PYG + ["dataloader.num_workers.training=8"])

In [ ]:
runs["workers_1"] = ph.run_experiment("workers_1", OVERRIDES + TRACE_ONLY + ["dataloader.num_workers.training=1"])

The speed report of each run is in `speed_profiler.csv`. The most important value is `training_avg_throughput`: the number of training steps for each second (it/s).

In [ ]:
pd.read_csv(runs["baseline"] / "speed_profiler.csv", index_col=0)

In [ ]:
pd.DataFrame(
    {
        "throughput [it/s]": {name: ph.training_throughput(d) for name, d in runs.items()},
        "data loader throughput [batches/s]": {name: ph.dataloader_throughput(d) for name, d in runs.items()},
    }
).round(2)

**Questions:**

1. How much slower is `pyg` than `baseline`? Why?
2. How much slower is `workers_1` than `baseline`? Compare the throughput of the training with the throughput of the data loader. What makes `workers_1` slow?

<details>
<summary>Answers</summary>

1. On this machine, it measured approximately 0.73 it/s with Triton and 0.40 it/s with PyG. PyG also uses more memory (we measured a peak of 6.4 GB, not 4.3 GB).
   The Triton kernel does the attention on the edges in one GPU kernel. PyG uses many kernels, and each kernel writes and reads large tensors for each edge in the GPU memory.
2. On this machine, it measured approximately 0.73 it/s with 1 worker. With 1 worker, one CPU process must prepare all the batches. If it is slower than the GPU, the GPU must wait for data. However, on these machines, loading the data is so quick that it is not a bottleneck.
   When the data loader throughput is near (or less than) the training throughput, the data loader is the bottleneck.
   Then, increase `dataloader.num_workers`, or put the dataset on a faster file system.
</details>

### 6.1 Open a trace in Perfetto

The profiler trace shows what the CPU and the GPU do at each moment of the 2 recorded training steps.
The cell below copies the training trace of each run into the folder `traces/`, next to this notebook.

In [ ]:
Path("traces").mkdir(exist_ok=True)
for name, profiler_dir in runs.items():
    target = Path("traces") / f"{name}.pt.trace.json"
    shutil.copy(ph.find_trace_file(profiler_dir), target)
    print(f"{target}  ({target.stat().st_size / 1e6:.0f} MB)")

1. In the JupyterLab file browser, open the folder `traces`. Right-click `baseline.pt.trace.json` and select **Download**. The file can be large, thus this can take some time.
2. On your laptop, open [https://ui.perfetto.dev](https://ui.perfetto.dev).
3. Click **Open trace file** (on the left) and select the file. You can also drag the file into the page.
4. Use the keys **W** and **S** to zoom in and out, and **A** and **D** to move left and right.

The trace has two important areas:

* the **CPU threads** (at the top): the Python code, and the operations that the CPU sends to the GPU
* the **GPU stream** (at the bottom, for example "stream 7"): the kernels that the GPU runs

**Exercise:**

1. Find the two `ProfilerStep#...` blocks on the CPU thread. Each block is one training step.
2. Look at the GPU stream. Are there gaps, where the GPU does nothing?
3. Click on a kernel in the GPU stream. Perfetto shows its name and its duration. Follow the arrow ("flow") to the CPU operation that started it.
4. Open `pyg.pt.trace.json` in a second tab. Search (at the top) for `scatter` or `index_select`. Then open `baseline.pt.trace.json` and search for `gt` or `triton`.
   Which trace has more GPU kernels? Which trace has longer steps?
5. Open `workers_1.pt.trace.json`. Look for long gaps on the GPU stream. On the CPU thread, what is the CPU doing during these gaps?
   (Tip: search for `DataLoader`.)

> **Tip:** A good training run has very few gaps on the GPU stream. A gap means that the GPU waits, usually for the CPU or for data.

<details>
<summary>Click for an example trace</summary>

![trace](images/trace_intro.png)

Follow a kernel back to the CPU:

![follow a kernel](images/trace_follow_kernel.png)
</details>

---
## Summary

| Tool or setting | What it does | When to use it |
|---|---|---|
| `diagnostics.print_memory_summary` | Prints the allocated and the reserved memory at the end of the training. | A quick look at the memory. |
| Memory snapshot (`benchmark_profiler.snapshot`) | Records each allocation, with its stack trace. Open it in [memory_viz](https://pytorch.org/memory_viz). | When you get `CUDA out of memory`, or to find the part of the model that makes the peak. |
| Profiler trace (`benchmark_profiler.memory`) | Records what the CPU and the GPU do. Open it in [Perfetto](https://ui.perfetto.dev). | When the model is slow, and you want to know why. |
| `graph_attention_backend: triton` | One fused GPU kernel for the graph attention. | Always, if your GPU supports Triton. |
| `num_chunks` | Calculates the mappers in parts. Less memory, a little slower. | When the mappers make the peak. Use 4 to 16. |
| `dataloader.num_workers` | CPU processes that prepare the batches. | Increase it if the data loader is slower than the training. |
| `num_gpus_per_model` | Shards one model over more GPUs. | When one sample does not fit on one GPU. |
| `num_gpus_per_ensemble` | Divides the ensemble members between GPUs. | Ensemble (CRPS) training with many members. |

### Further reading

* [anemoi performance optimisation guide](https://anemoi.readthedocs.io/projects/training/en/latest/user-guide/performance-optimisation.html)
* [anemoi benchmarking and profiler documentation](https://anemoi.readthedocs.io/projects/training/en/latest/user-guide/benchmarking.html)
* [PyTorch: understanding CUDA memory usage](https://pytorch.org/docs/stable/torch_cuda_memory.html)
* [A guide to the PyTorch CUDA caching allocator (blog)](https://zdevito.github.io/2022/08/04/cuda-caching-allocator.html)
* The long notebook `perf_profiling_long.ipynb` in this folder: more experiments, gradient checkpointing and higher resolutions.